In [46]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [47]:
BOOTSTRAP_SERVERS = "192.168.80.34:9092"

TOPIC_INPUT = "gittba10_LINK"
TOPIC_OUTPUT = "gittba10_link_VWAP2"

CHECKPOINT_KAFKA = "/tmp/gittba10_link_vwap2_kafka_checkpoint_v3"
CHECKPOINT_CONSOLE = "/tmp/gittba10_link_vwap2_console_checkpoint_v3"
CHECKPOINT_EVENTS = "/tmp/gittba10_link_events_checkpoint_v3"

In [48]:
conf = (SparkConf()
    .setMaster("yarn")
    .set("spark.executor.cores", 5)
    .set("spark.sql.shuffle.partitions", 200)
    .set("spark.default.parallelism", 200)
    .set("spark.executor.memory", "7g")
    .set("spark.dynamicAllocation.maxExecutors", 20)
    .set("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6")
)

spark = (SparkSession.builder
    .config(conf=conf)
    .appName("CryptoVWAPCalculator_LINK_VWAP2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.6


26/03/23 18:53:54 WARN SparkConf: The configuration key 'spark.blacklist.enabled' has been deprecated as of Spark 3.1.0 and may be removed in the future. Please use spark.excludeOnFailure.enabled
26/03/23 18:53:54 WARN SparkConf: The configuration key 'spark.yarn.max.executor.failures' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.maxNumFailures' instead.


In [49]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(spark.conf.get("spark.sql.session.timeZone"))

UTC


In [50]:
df_kafka_input = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", TOPIC_INPUT)
    .option("startingOffsets", "latest")
    .load()
)

In [35]:
input_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("@timestamp", StringType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True)
])

In [51]:
df_parsed = (df_kafka_input
    .selectExpr(
        "CAST(key AS STRING) AS kafka_key",
        "CAST(value AS STRING) AS kafka_value",
        "timestamp AS kafka_timestamp"
    )
    .withColumn("json_data", F.from_json(F.col("kafka_value"), input_schema))
    .select(
        F.col("kafka_key"),
        F.col("json_data.symbol").alias("symbol"),
        F.col("json_data.`@timestamp`").alias("event_time_str"),
        F.col("json_data.close").alias("close"),
        F.col("json_data.volume").alias("volume"),
        F.col("kafka_timestamp")
    )
)

-------------------------------------------
Batch: 45
-------------------------------------------
+--------+--------------------+-------------------+-----+------+
|symbol  |event_time_str      |event_time         |close|volume|
+--------+--------------------+-------------------+-----+------+
|LINKUSDT|2026-03-23T17:53:59Z|2026-03-23 17:53:59|9.12 |10.3  |
+--------+--------------------+-------------------+-----+------+



In [52]:
df_events = (df_parsed
    .withColumn(
        "event_time",
        F.to_timestamp("event_time_str", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .filter(F.col("symbol").isNotNull())
    .filter(F.col("close").isNotNull())
    .filter(F.col("volume").isNotNull())
    .filter(F.col("event_time").isNotNull())
)

[Stage 455:=========>  (162 + 38) / 200][Stage 457:>             (10 + 7) / 200]

In [53]:
# celda de depuración
query_events = (df_events
    .select("symbol", "event_time_str", "event_time", "close", "volume")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", CHECKPOINT_EVENTS)
    .start()
)

26/03/23 18:54:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/23 18:54:05 WARN StreamingQueryManager: Stopping existing streaming query [id=8fa1dffc-0329-4630-b3ab-ff79c358af3c, runId=1b1b65af-43e8-4140-b397-a79c7e86e34e], as a new run is being started.
26/03/23 18:54:06 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 79
-------------------------------------------
+-------------------+-------------------+--------+-------+-------+
|window_start       |window_end         |symbol  |vwap   |warning|
+-------------------+-------------------+--------+-------+-------+
|2026-03-23 17:50:00|2026-03-23 17:55:00|LINKUSDT|9.10845|NULL   |
+-------------------+-------------------+--------+-------+-------+



26/03/23 18:54:08 WARN TaskSetManager: Lost task 118.1 in stage 457.0 (TID 42107) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:54:08 WARN TaskSetManager: Lost task 138.1 in stage 457.0 (TID 42112) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:54:09 WARN TaskSetManager: Lost task 177.1 in stage 457.0 (TID 42141) (worker04.bigdata.alumnos.upcont.es executor 73): TaskKilled (another attempt succeeded)
26/03/23 18:54:09 WARN TaskSetManager: Lost task 198.1 in stage 457.0 (TID 42136) (worker01.bigdata.alumnos.upcont.es executor 58): TaskKilled (another attempt succeeded)
26/03/23 18:54:09 WARN TaskSetManager: Lost task 163.1 in stage 457.0 (TID 42135) (worker01.bigdata.alumnos.upcont.es executor 58): TaskKilled (another attempt succeeded)
26/03/23 18:54:10 WARN TaskSetManager: Lost task 176.1 in stage 457.0 (TID 42116) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (a

-------------------------------------------
Batch: 80
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



In [28]:
query_events.stop()

In [54]:
df_agg = (df_events
    .withWatermark("event_time", "5 minutes")
    .groupBy(
        F.window("event_time", "5 minutes"),
        F.col("symbol")
    )
    .agg(
        F.sum(F.col("close") * F.col("volume")).alias("sum_price_volume"),
        F.sum(F.col("volume")).alias("sum_volume")
    )
)

In [55]:
df_vwap = (df_agg
    .withColumn(
        "warning",
        F.when(
            F.col("sum_volume") == 0,
            F.lit("Volumen agregado igual a 0; no se puede calcular VWAP")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "vwap",
        F.when(
            F.col("sum_volume") != 0,
            F.col("sum_price_volume") / F.col("sum_volume")
        ).otherwise(F.lit(None))
    )
)

In [56]:
df_console = (df_vwap
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("symbol"),
        F.round(F.col("vwap"), 5).alias("vwap"),
        F.col("warning")
    )
)

In [57]:
# otra celda de validación
query_console = (df_console
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", CHECKPOINT_CONSOLE)
    .start()
)

26/03/23 18:54:35 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/23 18:54:35 WARN StreamingQueryManager: Stopping existing streaming query [id=ff7f106d-f934-4708-9d7f-fd1b703a27ae, runId=5f37e96a-97d5-4c51-95de-a7a0fcad0533], as a new run is being started.
26/03/23 18:54:35 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 46
-------------------------------------------
+--------+--------------------+-------------------+-----+-------+
|symbol  |event_time_str      |event_time         |close|volume |
+--------+--------------------+-------------------+-----+-------+
|LINKUSDT|2026-03-23T17:54:59Z|2026-03-23 17:54:59|9.11 |5536.61|
+--------+--------------------+-------------------+-----+-------+



26/03/23 18:55:07 WARN TaskSetManager: Lost task 148.1 in stage 466.0 (TID 42898) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:55:07 WARN TaskSetManager: Lost task 153.1 in stage 466.0 (TID 42902) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:55:07 WARN TaskSetManager: Lost task 151.1 in stage 466.0 (TID 42897) (worker04.bigdata.alumnos.upcont.es executor 71): TaskKilled (another attempt succeeded)
26/03/23 18:55:07 WARN TaskSetManager: Lost task 157.1 in stage 466.0 (TID 42907) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:55:07 WARN TaskSetManager: Lost task 175.1 in stage 466.0 (TID 42906) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:55:08 WARN TaskSetManager: Lost task 177.1 in stage 466.0 (TID 42901) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (a

-------------------------------------------
Batch: 81
-------------------------------------------
+-------------------+-------------------+--------+-------+-------+
|window_start       |window_end         |symbol  |vwap   |warning|
+-------------------+-------------------+--------+-------+-------+
|2026-03-23 17:50:00|2026-03-23 17:55:00|LINKUSDT|9.10917|NULL   |
+-------------------+-------------------+--------+-------+-------+



26/03/23 18:55:14 WARN TaskSetManager: Lost task 184.1 in stage 470.0 (TID 43302) (worker02.bigdata.alumnos.upcont.es executor 35): TaskKilled (another attempt succeeded)
                                                                                

-------------------------------------------
Batch: 82
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+

-------------------------------------------
Batch: 47
-------------------------------------------
+--------+--------------------+-------------------+-----+-------+
|symbol  |event_time_str      |event_time         |close|volume |
+--------+--------------------+-------------------+-----+-------+
|LINKUSDT|2026-03-23T17:55:59Z|2026-03-23 17:55:59|9.12 |1611.78|
+--------+--------------------+-------------------+-----+-------+



-------------------------------------------
Batch: 83
-------------------------------------------
+-------------------+-------------------+--------+----+-------+
|window_start       |window_end         |symbol  |vwap|warning|
+-------------------+-------------------+--------+----+-------+
|2026-03-23 17:55:00|2026-03-23 18:00:00|LINKUSDT|9.12|NULL   |
+-------------------+-------------------+--------+----+-------+



26/03/23 18:56:14 WARN TaskSetManager: Lost task 171.1 in stage 477.0 (TID 44009) (worker04.bigdata.alumnos.upcont.es executor 72): TaskKilled (another attempt succeeded)
26/03/23 18:56:14 WARN TaskSetManager: Lost task 168.1 in stage 477.0 (TID 44010) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:56:14 WARN TaskSetManager: Lost task 174.1 in stage 477.0 (TID 44056) (worker04.bigdata.alumnos.upcont.es executor 72): TaskKilled (another attempt succeeded)
26/03/23 18:56:14 WARN TaskSetManager: Lost task 179.1 in stage 477.0 (TID 44074) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:56:14 WARN TaskSetManager: Lost task 177.1 in stage 477.0 (TID 44075) (worker04.bigdata.alumnos.upcont.es executor 73): TaskKilled (another attempt succeeded)
26/03/23 18:56:14 WARN TaskSetManager: Lost task 197.1 in stage 477.0 (TID 44079) (worker04.bigdata.alumnos.upcont.es executor 73): TaskKilled (S

-------------------------------------------
Batch: 84
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



In [58]:
df_output_kafka = (df_vwap
    .select(
        F.col("symbol").alias("key"),
        F.to_json(
            F.struct(
                F.date_format(F.col("window.start"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("window_start"),
                F.date_format(F.col("window.end"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("window_end"),
                F.col("symbol").alias("symbol"),
                F.round(F.col("vwap"), 5).alias("vwap"),
                F.col("warning").alias("warning")
            )
        ).alias("value")
    )
)

In [59]:
query_kafka = (df_output_kafka
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("topic", TOPIC_OUTPUT)
    .option("checkpointLocation", CHECKPOINT_KAFKA)
    .outputMode("append")
    .start()
)

26/03/23 18:56:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/23 18:56:50 WARN StreamingQueryManager: Stopping existing streaming query [id=046e347a-7fde-4975-9264-284d5ff3c782, runId=7f3b1f87-5b65-4d2f-b418-98138ecd6b32], as a new run is being started.
26/03/23 18:56:50 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 48
-------------------------------------------
+--------+--------------------+-------------------+-----+------+
|symbol  |event_time_str      |event_time         |close|volume|
+--------+--------------------+-------------------+-----+------+
|LINKUSDT|2026-03-23T17:56:59Z|2026-03-23 17:56:59|9.13 |130.1 |
+--------+--------------------+-------------------+-----+------+



-------------------------------------------
Batch: 85
-------------------------------------------
+-------------------+-------------------+--------+-------+-------+
|window_start       |window_end         |symbol  |vwap   |warning|
+-------------------+-------------------+--------+-------+-------+
|2026-03-23 17:55:00|2026-03-23 18:00:00|LINKUSDT|9.12075|NULL   |
+-------------------+-------------------+--------+-------+-------+



26/03/23 18:57:08 WARN TaskSetManager: Lost task 190.1 in stage 484.0 (TID 44563) (worker02.bigdata.alumnos.upcont.es executor 41): TaskKilled (another attempt succeeded)
26/03/23 18:57:09 WARN TaskSetManager: Lost task 192.1 in stage 484.0 (TID 44568) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 18:57:11 WARN TaskSetManager: Lost task 194.0 in stage 484.0 (TID 44509) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (Finish but did not commit due to another attempt succeeded)
26/03/23 18:57:11 WARN TaskSetManager: Lost task 197.0 in stage 484.0 (TID 44512) (worker02.bigdata.alumnos.upcont.es executor 40): TaskKilled (another attempt succeeded)
26/03/23 18:57:11 WARN TaskSetManager: Lost task 199.1 in stage 484.0 (TID 44573) (worker02.bigdata.alumnos.upcont.es executor 28): TaskKilled (another attempt succeeded)
26/03/23 18:57:11 WARN TaskSetManager: Lost task 195.0 in stage 484.0 (TID 44510) (worker02.bigdata.alumnos.upco

-------------------------------------------
Batch: 86
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



...

-------------------------------------------
Batch: 57
-------------------------------------------
+--------+--------------------+-------------------+-----+------+
|symbol  |event_time_str      |event_time         |close|volume|
+--------+--------------------+-------------------+-----+------+
|LINKUSDT|2026-03-23T18:05:59Z|2026-03-23 18:05:59|9.13 |2977.1|
+--------+--------------------+-------------------+-----+------+



26/03/23 19:06:07 WARN TaskSetManager: Lost task 104.1 in stage 565.0 (TID 51894) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 19:06:07 WARN TaskSetManager: Lost task 102.1 in stage 565.0 (TID 51895) (worker04.bigdata.alumnos.upcont.es executor 80): TaskKilled (another attempt succeeded)
26/03/23 19:06:07 WARN TaskSetManager: Lost task 101.1 in stage 565.0 (TID 51896) (worker04.bigdata.alumnos.upcont.es executor 81): TaskKilled (another attempt succeeded)
26/03/23 19:06:07 WARN TaskSetManager: Lost task 188.1 in stage 565.0 (TID 51893) (worker02.bigdata.alumnos.upcont.es executor 28): TaskKilled (another attempt succeeded)
26/03/23 19:06:07 WARN TaskSetManager: Lost task 198.1 in stage 563.0 (TID 51817) (worker04.bigdata.alumnos.upcont.es executor 81): TaskKilled (Stage finished)
26/03/23 19:06:07 WARN TaskSetManager: Lost task 164.1 in stage 563.0 (TID 51816) (worker01.bigdata.alumnos.upcont.es executor 82): TaskKilled (Stage finish

-------------------------------------------
Batch: 103
-------------------------------------------
+-------------------+-------------------+--------+----+-------+
|window_start       |window_end         |symbol  |vwap|warning|
+-------------------+-------------------+--------+----+-------+
|2026-03-23 18:05:00|2026-03-23 18:10:00|LINKUSDT|9.13|NULL   |
+-------------------+-------------------+--------+----+-------+



26/03/23 19:06:12 WARN TaskSetManager: Lost task 174.1 in stage 569.0 (TID 52269) (worker04.bigdata.alumnos.upcont.es executor 74): TaskKilled (another attempt succeeded)
26/03/23 19:06:12 WARN TaskSetManager: Lost task 169.1 in stage 569.0 (TID 52270) (worker04.bigdata.alumnos.upcont.es executor 80): TaskKilled (another attempt succeeded)
26/03/23 19:06:12 WARN TaskSetManager: Lost task 195.1 in stage 569.0 (TID 52294) (worker01.bigdata.alumnos.upcont.es executor 82): TaskKilled (Stage finished)


-------------------------------------------
Batch: 104
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



In [45]:
query_console.awaitTermination()
query_kafka.awaitTermination()

-------------------------------------------
Batch: 10
-------------------------------------------
+--------+--------------------+-------------------+-----+-------+
|symbol  |event_time_str      |event_time         |close|volume |
+--------+--------------------+-------------------+-----+-------+
|LINKUSDT|2026-03-23T17:18:59Z|2026-03-23 17:18:59|9.05 |4870.77|
+--------+--------------------+-------------------+-----+-------+



-------------------------------------------
Batch: 9
-------------------------------------------
+-------------------+-------------------+--------+-------+-------+
|window_start       |window_end         |symbol  |vwap   |warning|
+-------------------+-------------------+--------+-------+-------+
|2026-03-23 17:15:00|2026-03-23 17:20:00|LINKUSDT|9.05057|NULL   |
+-------------------+-------------------+--------+-------+-------+



26/03/23 18:19:23 WARN TaskSetManager: Lost task 181.1 in stage 142.0 (TID 13537) (worker02.bigdata.alumnos.upcont.es executor 26): TaskKilled (Stage finished)
26/03/23 18:19:23 WARN TaskSetManager: Lost task 183.1 in stage 142.0 (TID 13539) (worker02.bigdata.alumnos.upcont.es executor 26): TaskKilled (Stage finished)
26/03/23 18:19:23 WARN TaskSetManager: Lost task 184.1 in stage 142.0 (TID 13538) (worker02.bigdata.alumnos.upcont.es executor 28): TaskKilled (Stage finished)
26/03/23 18:19:23 WARN TaskSetManager: Lost task 182.1 in stage 142.0 (TID 13540) (worker02.bigdata.alumnos.upcont.es executor 28): TaskKilled (Stage finished)
                                                                                

-------------------------------------------
Batch: 10
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



26/03/23 18:19:30 WARN TaskSetManager: Lost task 147.1 in stage 146.0 (TID 13842) (worker04.bigdata.alumnos.upcont.es executor 30): TaskKilled (Stage finished)
26/03/23 18:19:30 WARN TaskSetManager: Lost task 148.1 in stage 146.0 (TID 13844) (worker02.bigdata.alumnos.upcont.es executor 28): TaskKilled (Stage finished)
26/03/23 18:19:30 WARN TaskSetManager: Lost task 149.1 in stage 146.0 (TID 13843) (worker02.bigdata.alumnos.upcont.es executor 31): TaskKilled (Stage finished)
26/03/23 18:19:30 WARN TaskSetManager: Lost task 180.1 in stage 146.0 (TID 13845) (worker02.bigdata.alumnos.upcont.es executor 27): TaskKilled (Stage finished)


...

-------------------------------------------
Batch: 77
-------------------------------------------
+-------------------+-------------------+--------+-------+-------+
|window_start       |window_end         |symbol  |vwap   |warning|
+-------------------+-------------------+--------+-------+-------+
|2026-03-23 17:50:00|2026-03-23 17:55:00|LINKUSDT|9.10844|NULL   |
+-------------------+-------------------+--------+-------+-------+



26/03/23 18:53:13 WARN TaskSetManager: Lost task 196.0 in stage 452.0 (TID 41569) (worker04.bigdata.alumnos.upcont.es executor 72): TaskKilled (Finish but did not commit due to another attempt succeeded)
26/03/23 18:53:13 WARN TaskSetManager: Lost task 111.1 in stage 452.0 (TID 41660) (worker04.bigdata.alumnos.upcont.es executor 72): TaskKilled (another attempt succeeded)
26/03/23 18:53:14 WARN TaskSetManager: Lost task 108.1 in stage 452.0 (TID 41661) (worker01.bigdata.alumnos.upcont.es executor 58): TaskKilled (Stage finished)


-------------------------------------------
Batch: 78
-------------------------------------------
+------------+----------+------+----+-------+
|window_start|window_end|symbol|vwap|warning|
+------------+----------+------+----+-------+
+------------+----------+------+----+-------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/sharenfs/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/sharenfs/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/share/anaconda/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
query_console.stop()
query_kafka.stop()

In [ ]:
# parar celda de depuracion
query_events.stop()
query_console.stop()
query_kafka.stop()